# Cross-Context Aggregation Consistency Test

In [9]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

In [10]:
df = pd.read_csv('../results/all_responses.csv')

CONTEXTS_SEPARATE = ['academic_support', 'attendance_issue', 'behavioral_concern', 'discipline_referral']

ALPHA_THRESHOLD = 0.6

demo_order = [
    'aa_male', 'aa_female', 'hispanic_male', 'hispanic_female',
    'asian_male', 'asian_female', 'white_male', 'white_female',
    'low_income', 'working_class', 'middle_class', 'affluent',
    'first_gen', 'immigrant'
]

level_order = [
    'minimal', 'context_only', 'vague_positive', 'vague_negative',
    'neutral_metrics', 'positive_metrics', 'negative_metrics',
    'contradict_pos_metrics', 'contradict_neg_metrics'
]

In [11]:
sep_df = df[df['context'].isin(CONTEXTS_SEPARATE)].copy()

question_context_map = (
    sep_df.groupby('question_key')['context']
    .apply(lambda x: sorted(x.unique().tolist()))
    .to_dict()
)

for q, ctxs in sorted(question_context_map.items()):
    can_test = len(ctxs) >= 2
    tag = 'testable' if can_test else 'single context'
    print(f'  {q:<30s} ({len(ctxs)} contexts) {tag}')
    for c in ctxs:
        print(f'      - {c}')

CROSS_QUESTIONS = {q: ctxs for q, ctxs in question_context_map.items() if len(ctxs) >= 2}

  discipline_severity            (1 contexts) single context
      - discipline_referral
  effort_attribution             (4 contexts) testable
      - academic_support
      - attendance_issue
      - behavioral_concern
      - discipline_referral
  intervention_priority          (2 contexts) testable
      - attendance_issue
      - behavioral_concern
  parent_engagement              (1 contexts) single context
      - behavioral_concern
  potential_assessment           (1 contexts) single context
      - academic_support
  success_likelihood             (1 contexts) single context
      - academic_support
  trust_explanation              (2 contexts) testable
      - attendance_issue
      - discipline_referral


In [12]:
def cronbach_alpha(item_scores):
    """Compute Cronbach's alpha. Columns = items, rows = observations."""
    item_scores = item_scores.dropna(axis=1, how='all').dropna(axis=0, how='any')
    if item_scores.shape[1] < 2:
        return np.nan
    k = item_scores.shape[1]
    item_variances = item_scores.var(axis=0, ddof=1)
    total_variance = item_scores.sum(axis=1).var(ddof=1)
    if total_variance == 0:
        return np.nan
    return (k / (k - 1)) * (1 - item_variances.sum() / total_variance)

In [13]:
results_raw = []

for question, contexts in CROSS_QUESTIONS.items():
    for model in sorted(df['model'].unique()):
        mask = (
            (sep_df['question_key'] == question) &
            (sep_df['context'].isin(contexts)) &
            (sep_df['model'] == model)
        )
        q_df = sep_df[mask]
        
        if len(q_df) == 0:
            continue
        
        agg_df = q_df.groupby(['demographic_id', 'level', 'context'])['score'].mean().reset_index()
        
        pivot = agg_df.pivot_table(
            index=['demographic_id', 'level'],
            columns='context',
            values='score'
        )
        
        alpha = cronbach_alpha(pivot)
        n_obs = pivot.dropna().shape[0]
        n_items = pivot.dropna(axis=1, how='all').shape[1]
        
        corr_matrix = pivot.corr()
        mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        pairwise_corrs = corr_matrix.where(mask_upper).stack()
        mean_r = pairwise_corrs.mean() if len(pairwise_corrs) > 0 else np.nan
        
        results_raw.append({
            'question': question,
            'model': model,
            'n_contexts': n_items,
            'n_observations': n_obs,
            'cronbach_alpha': alpha,
            'mean_inter_context_r': mean_r
        })

results_df = pd.DataFrame(results_raw)


In [14]:
for question in sorted(CROSS_QUESTIONS.keys()):
    q_results = results_df[results_df['question'] == question]
    ctxs = CROSS_QUESTIONS[question]
    print(f'\n{"─" * 85}')
    print(f'Question: {question}')
    print(f'Contexts: {" + ".join(ctxs)} ({len(ctxs)} items)')
    print(f'{"─" * 85}')
    print(f'{"Model":<25s} {"α":>8s} {"Mean r":>8s} {"N obs":>8s} {"Pass?":>8s}')
    print(f'{"-" * 25} {"-" * 8} {"-" * 8} {"-" * 8} {"-" * 8}')
    
    for _, row in q_results.sort_values('model').iterrows():
        alpha = row['cronbach_alpha']
        passes = 'pass' if (pd.notna(alpha) and alpha >= ALPHA_THRESHOLD) else 'fails'
        alpha_str = f'{alpha:.3f}' if pd.notna(alpha) else 'N/A'
        mean_r_str = f'{row["mean_inter_context_r"]:.3f}' if pd.notna(row["mean_inter_context_r"]) else 'N/A'
        print(f'{row["model"]:<25s} {alpha_str:>8s} {mean_r_str:>8s} '
              f'{row["n_observations"]:>8.0f} {passes:>8s}')
    
    mean_alpha = q_results['cronbach_alpha'].mean()
    mean_r = q_results['mean_inter_context_r'].mean()
    n_pass = (q_results['cronbach_alpha'] >= ALPHA_THRESHOLD).sum()
    mean_alpha_str = f'{mean_alpha:.3f}' if pd.notna(mean_alpha) else 'N/A'
    mean_r_str = f'{mean_r:.3f}' if pd.notna(mean_r) else 'N/A'
    print(f'{"MEAN":.<25s} {mean_alpha_str:>8s} {mean_r_str:>8s} {"":>8s} {n_pass}/{len(q_results)}')


─────────────────────────────────────────────────────────────────────────────────────
Question: effort_attribution
Contexts: academic_support + attendance_issue + behavioral_concern + discipline_referral (4 items)
─────────────────────────────────────────────────────────────────────────────────────
Model                            α   Mean r    N obs    Pass?
------------------------- -------- -------- -------- --------
deepseek-chat                0.501    0.277      135    fails
deepseek-reasoner            0.723    0.395      135     pass
gemini-flash                 0.565    0.288      135    fails
gpt-5                        0.621    0.354      135     pass
gpt-5-nano                   0.756    0.511      135     pass
grok-4-1-fast                0.697    0.384      135     pass
MEAN.....................    0.644    0.368          4/6

─────────────────────────────────────────────────────────────────────────────────────
Question: intervention_priority
Contexts: attendance_issue 

In [15]:
summary = results_df.groupby('question').agg(
    n_contexts=('n_contexts', 'first'),
    alpha_mean=('cronbach_alpha', 'mean'),
    alpha_min=('cronbach_alpha', 'min'),
    alpha_max=('cronbach_alpha', 'max'),
    r_mean=('mean_inter_context_r', 'mean'),
    models_passing=('cronbach_alpha', lambda x: (x >= ALPHA_THRESHOLD).sum()),
    models_total=('cronbach_alpha', 'count')
).round(3)

summary['pass_rate'] = (summary['models_passing'].astype(str) + '/' + summary['models_total'].astype(str))
summary['aggregate?'] = summary['alpha_mean'].apply(
    lambda x: 'pass' if x >= ALPHA_THRESHOLD else 'fail'
)

display_cols = ['n_contexts', 'alpha_mean', 'alpha_min', 'alpha_max', 'r_mean', 'pass_rate', 'aggregate?']
print(summary[display_cols].rename(columns={
    'n_contexts': 'Contexts',
    'alpha_mean': '\u03b1 mean',
    'alpha_min': '\u03b1 min',
    'alpha_max': '\u03b1 max',
    'r_mean': 'Mean r',
    'pass_rate': 'Models pass',
    'aggregate?': 'Decision'
}).to_string())

print()
passing = summary[summary['alpha_mean'] >= ALPHA_THRESHOLD].index.tolist()
failing = summary[summary['alpha_mean'] < ALPHA_THRESHOLD].index.tolist()

                       Contexts  α mean  α min  α max  Mean r Models pass Decision
question                                                                          
effort_attribution            4   0.644  0.501  0.756   0.368         4/6     pass
intervention_priority         2   0.822  0.531  0.950   0.719         5/6     pass
trust_explanation             2   0.668  0.370  0.866   0.622         4/6     pass

